# 22.8 实验追踪:MLflow 与可复现性 / Experiment Tracking (MLflow) & Reproducibility

**中文**:做机器学习,你会跑**成百上千次实验**:换个超参、加个特征、改个模型。三周后老板问:"上次那个准确率 0.92 的模型,是用什么参数训的?" 如果你只是在 notebook 里手动改数字、结果记在脑子里或散落的 txt 里,答案往往是——**"我不记得了"**。这就是**实验追踪(experiment tracking)** 要解决的核心痛点:*系统性地记录每一次实验的参数、指标、模型、数据版本,让实验可比较、可复现、可追溯*。**MLflow** 是这个领域的事实标准。本节从零实现一个 mini 实验追踪器(记录参数/指标/模型产物、比较所有 run、自动挑最优),让你看清它的本质其实很朴素,再给出真实 MLflow 代码。这是 22.7 质量门"可复现"前提的落地工具。
**English**: In machine learning you run **hundreds or thousands of experiments**: change a hyperparameter, add a feature, swap a model. Three weeks later your boss asks: "That 0.92-accuracy model from last time — what parameters trained it?" If you just manually edited numbers in a notebook and kept results in your head or scattered txt files, the answer is often — **"I don't remember."** This is the core pain **experiment tracking** solves: *systematically log every experiment's parameters, metrics, model, and data version, making experiments comparable, reproducible, and traceable*. **MLflow** is this field's de-facto standard. This section builds a mini experiment tracker from scratch (log params/metrics/model artifacts, compare all runs, auto-pick the best), showing its essence is actually plain, then gives real MLflow code. It's the implementation tool for 22.7's quality gate "reproducibility" prerequisite.

---

**中文**:**实验追踪记录什么(每次 run)**:
**English**: **What experiment tracking logs (per run)**:
- **中文**:**参数(params)**:超参数、配置(学习率、树的数量、特征列表、数据版本)——**"你怎么训的"**。
  **Parameters**: hyperparameters, config (learning rate, number of trees, feature list, data version) — **"how you trained it."**
- **中文**:**指标(metrics)**:评估结果(准确率、F1、loss 曲线)——**"训得怎么样"**。
  **Metrics**: evaluation results (accuracy, F1, loss curves) — **"how well it trained."**
- **中文**:**产物(artifacts)**:训好的模型文件、图表、混淆矩阵——**"训出了什么"**。
  **Artifacts**: the trained model file, charts, confusion matrix — **"what it produced."**
- **中文**:**元数据**:代码版本(git commit)、时间、运行环境——**可追溯**。
  **Metadata**: code version (git commit), time, run environment — **traceable**.

**中文**:有了这些,你就能:①**比较**几百次 run,一眼找出最优;②**复现**任何历史模型(知道确切参数+数据+代码);③**追溯**"这个上线的模型当初是怎么来的"。**MLflow 还提供模型注册表(Model Registry)**:给模型版本化、管理阶段流转(Staging → Production)、记录血缘——这直接支撑 22.7 的质量门和回滚。
**English**: With these you can: ① **compare** hundreds of runs and instantly find the best; ② **reproduce** any historical model (knowing exact params + data + code); ③ **trace** "how did this deployed model come to be." **MLflow also provides a Model Registry**: version models, manage stage transitions (Staging → Production), record lineage — directly supporting 22.7's quality gate and rollback.

> 💡 **面试速查 / Interview cheat-sheet（★★ MLOps 必考）**
> **中文**:**实验追踪**=系统记录每次 run 的**参数+指标+模型产物+元数据(代码/数据版本)**→实验可比较、可复现、可追溯(解决"忘了那个好模型怎么训的")。**MLflow 四大组件**:①**Tracking**(记录 params/metrics/artifacts, UI 比较 run)②**Model Registry**(模型版本化+阶段流转 Staging/Production+血缘, 接 22.7 质量门/回滚)③**Projects**(打包可复现运行)④**Models**(统一模型打包格式, 多引擎部署)。**可复现三要素**:固定随机种子 + 钉数据版本(DVC)+ 钉代码版本(git commit)+ 钉环境(容器)。**vs 别的**:**Weights & Biases**(更强的可视化/协作/大模型训练监控)、**Neptune**、**Comet**、**DVC**(偏数据/流水线版本)。**用途**:超参搜索对比、团队共享实验、模型血缘审计、和 CI/CD(22.7)/特征库(22.9)打通。面试金句:*"实验追踪(MLflow)系统记录每次实验的参数、指标、模型产物和代码/数据版本, 让几百次实验可比较、可复现、可追溯; MLflow 还有模型注册表做版本化和 Staging→Production 阶段管理, 支撑质量门和回滚; 可复现要固定种子+钉数据/代码/环境版本; 大模型训练监控常用 W&B。"*
> **English**: **Experiment tracking** = systematically log every run's **params + metrics + model artifacts + metadata (code/data version)** → experiments become comparable, reproducible, traceable (solving "forgot how that good model was trained"). **MLflow's four components**: ① **Tracking** (log params/metrics/artifacts, UI to compare runs) ② **Model Registry** (version models + stage transitions Staging/Production + lineage, ties to 22.7's quality gate/rollback) ③ **Projects** (package reproducible runs) ④ **Models** (unified model packaging format, multi-engine deployment). **Reproducibility's essentials**: fix random seeds + pin data version (DVC) + pin code version (git commit) + pin environment (container). **vs others**: **Weights & Biases** (stronger visualization/collaboration/large-model training monitoring), **Neptune**, **Comet**, **DVC** (leans data/pipeline versioning). **Uses**: hyperparameter search comparison, team-shared experiments, model lineage audit, integration with CI/CD (22.7)/feature store (22.9). Interview line: *"Experiment tracking (MLflow) systematically logs each experiment's parameters, metrics, model artifacts, and code/data versions, making hundreds of experiments comparable, reproducible, traceable; MLflow also has a model registry for versioning and Staging→Production stage management, supporting the quality gate and rollback; reproducibility needs fixed seeds + pinned data/code/environment versions; large-model training monitoring often uses W&B."*


In [ ]:

# ============================================================
# 从零实现一个 mini 实验追踪器(复现 mlflow 核心)/ mini experiment tracker from scratch
# 中文:mlflow 本机没装。我们用纯 Python 复现它的核心:每次 run 记录 params/metrics/artifacts 到一个存储,
#      之后能查询、比较所有 run、挑最优。真实 MLflow 代码见下方。
# English: mlflow isn't installed; we reproduce its core in pure Python: each run logs params/metrics/artifacts to a
#      store, then we query, compare all runs, and pick the best. Real MLflow code below.
# ============================================================
import os, json, time, shutil, hashlib, numpy as np, joblib, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
np.random.seed(0)
STORE="/tmp/mlruns"
if os.path.exists(STORE): shutil.rmtree(STORE)
os.makedirs(STORE)

class Run:                                                 # 一次实验 run(对标 mlflow.start_run)/ one run (like mlflow.start_run)
    def __init__(self, exp): self.id=hashlib.md5(str(time.time_ns()).encode()).hexdigest()[:8]; self.d={"exp":exp,"params":{},"metrics":{}}
    def __enter__(self): return self
    def log_param(self,k,v):  self.d["params"][k]=v         # 记录超参 / log a parameter
    def log_metric(self,k,v): self.d["metrics"][k]=round(float(v),4)   # 记录指标 / log a metric
    def log_model(self,model):                             # 记录模型产物 / log the model artifact
        os.makedirs(f"{STORE}/{self.id}",exist_ok=True); joblib.dump(model,f"{STORE}/{self.id}/model.joblib")
    def __exit__(self,*a):
        os.makedirs(f"{STORE}/{self.id}",exist_ok=True); json.dump(self.d,open(f"{STORE}/{self.id}/meta.json","w"))
def search_runs(exp):                                      # 查询某实验的所有 run(对标 mlflow.search_runs)/ query all runs
    out=[]
    for rid in os.listdir(STORE):
        m=json.load(open(f"{STORE}/{rid}/meta.json"))
        if m["exp"]==exp: out.append({"run":rid,**m["params"],**m["metrics"]})
    return out

X,y=make_classification(n_samples=2000,n_features=20,n_informative=10,random_state=0)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,random_state=0)
# 一个被追踪的超参搜索:每个组合是一次 run / a tracked hyperparameter sweep: each combo is a run
for n_est in [10,50,200]:
    for depth in [3,None]:
        with Run("rf_sweep") as run:
            m=RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=0).fit(Xtr,ytr)
            pred=m.predict(Xte)
            run.log_param("n_estimators",n_est); run.log_param("max_depth",str(depth))     # 记参数 / log params
            run.log_metric("accuracy",accuracy_score(yte,pred)); run.log_metric("f1",f1_score(yte,pred))  # 记指标 / log metrics
            run.log_model(m)                                                                # 记模型 / log model
print(f"追踪了 {len(search_runs('rf_sweep'))} 次实验 run。所有 run 一览(对标 MLflow UI)/ all runs:")
print(f"{'run':>10}{'n_est':>7}{'depth':>7}{'accuracy':>10}{'f1':>8}")
for r in sorted(search_runs("rf_sweep"), key=lambda r:-r["accuracy"]):
    print(f"{r['run']:>10}{r['n_estimators']:>7}{r['max_depth']:>7}{r['accuracy']:>10}{r['f1']:>8}")
best=max(search_runs("rf_sweep"), key=lambda r:r["accuracy"])
print(f"\n自动挑出最优 run: {best['run']} (n_estimators={best['n_estimators']}, max_depth={best['max_depth']}, acc={best['accuracy']})")
print("→ 每次实验的参数+指标+模型全被记录, 三周后也能精确复现'那个 0.92 的模型'")


**中文**:上面从零实现了追踪器。下面是**真实的 MLflow 代码**——生产里就这么写(注意它把我们手写的记录逻辑变成几个 API 调用):
**English**: The above implements a tracker from scratch. Below is **real MLflow code** — this is how you'd write it in production (note it turns our hand-written logging into a few API calls):

```python
import mlflow
from mlflow.models import infer_signature

mlflow.set_experiment("rf_sweep")                 # 一个实验 = 一组相关 run / an experiment = a group of related runs

for n_est in [10, 50, 200]:
    for depth in [3, None]:
        with mlflow.start_run():                  # 开一个 run(自动记时间/git commit)/ start a run (auto-logs time/git)
            model = RandomForestClassifier(n_estimators=n_est, max_depth=depth).fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))

            mlflow.log_param("n_estimators", n_est)          # 记参数 / log params
            mlflow.log_param("max_depth", depth)
            mlflow.log_metric("accuracy", acc)               # 记指标 / log metrics
            mlflow.sklearn.log_model(model, "model",         # 记模型(带 signature, 便于部署)/ log model
                                     signature=infer_signature(X_train, model.predict(X_train)))

# 查询 + 挑最优 + 注册到 Model Registry(阶段流转 Staging→Production, 接 22.7)/ query + best + register
best = mlflow.search_runs(order_by=["metrics.accuracy DESC"]).iloc[0]
mlflow.register_model(f"runs:/{best.run_id}/model", "iris-classifier")   # 版本化 + 血缘 / versioning + lineage
```
**中文**:然后 `mlflow ui` 就能打开一个网页,把所有 run 的参数/指标排成表、画成图、并排比较——这就是 22.7 质量门里"和生产基线比较"的数据来源。
**English**: Then `mlflow ui` opens a web page tabulating and charting all runs' params/metrics for side-by-side comparison — the data source for 22.7's quality gate "compare against the production baseline."


In [ ]:

# ============================================================
# 可视化:实验对比 + 可复现验证 / experiment comparison + reproducibility check
# ============================================================
import matplotlib.pyplot as plt
runs=sorted(search_runs("rf_sweep"), key=lambda r:r["accuracy"])
labels=[f"n={r['n_estimators']}\nd={r['max_depth']}" for r in runs]; accs=[r["accuracy"] for r in runs]
fig,ax=plt.subplots(1,2,figsize=(14,5))
cols=["#C44E52" if a<0.85 else "#55A868" for a in accs]
b=ax[0].barh(labels, accs, color=cols)
for bar,a in zip(b,accs): ax[0].text(a+0.003,bar.get_y()+bar.get_height()/2,f"{a:.3f}",va="center",fontsize=9)
ax[0].axvline(0.85,ls="--",color="gray",alpha=0.6); ax[0].set_xlim(0.7,0.95)
ax[0].set_xlabel("accuracy"); ax[0].set_title("所有实验 run 一览(MLflow UI 就是这个)")
# 可复现:用记录的最优参数重训, 得到完全相同的结果 / reproduce best run from logged params
import joblib
best=max(search_runs("rf_sweep"), key=lambda r:r["accuracy"])
reloaded=joblib.load(f"{STORE}/{best['run']}/model.joblib")           # 直接加载记录的模型产物 / load logged artifact
retrained=RandomForestClassifier(n_estimators=best["n_estimators"],
            max_depth=None if best["max_depth"]=="None" else int(best["max_depth"]), random_state=0).fit(Xtr,ytr)
same=np.array_equal(reloaded.predict(Xte), retrained.predict(Xte))
ax[1].axis("off"); ax[1].set_title("可复现性:凭记录的参数完美重建最优模型",fontsize=12,weight="bold")
ax[1].text(0.5,0.6,f"最优 run: {best['run']}\n参数: n_estimators={best['n_estimators']}, max_depth={best['max_depth']}\n\n"
                   f"用记录的参数重新训练\n→ 与记录的模型预测完全一致: {same}",
           ha="center",va="center",fontsize=11,family="monospace",
           bbox=dict(boxstyle="round",fc="#55A868",alpha=0.15),transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/mlops08_viz.png",dpi=80); plt.show()
print(f"因为参数被记录, 三周后也能凭它精确重建最优模型(重训与记录一致={same})——这就是可复现性")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **实验追踪的本质朴素得惊人,但没有它整个 ML 工作流会崩溃**:我们几十行就复现了 MLflow 的核心——不过是"每次 run 把参数、指标、模型存进一个有结构的地方,之后能查询比较"。但正是这个朴素的东西,把 ML 从"炼丹式的手工作坊"变成"可管理的工程"。没有它,你会陷入所有数据科学家都经历过的地狱:notebook 里手改数字、结果记在便利贴上、三周后完全不记得那个最好的模型是怎么来的、想复现却复现不出、团队成员重复跑一样的实验……**实验追踪是把"我试了很多"变成"我知道我试了什么、哪个最好、为什么"的关键。**
2. **它的深层价值是"可复现性",这是科学和工程的底线**:我们验证了——因为参数被完整记录,三周后(甚至换个人)都能**凭记录精确重建那个最优模型**。可复现性听起来理所当然,实则是 ML 最容易失守的地方:随机种子没固定、数据悄悄变了、依赖版本升级了、某个预处理步骤忘了记……任何一个都会让"上次的 0.92 这次变 0.89",而你无从判断是改进还是噪声。实验追踪 + 数据版本化(DVC)+ 代码版本(git)+ 环境固化(容器)一起,构成可复现的四根支柱——这也正是 22.7 质量门能成立的前提(要"和基线比较",基线本身必须可复现可信)。
3. **诚实的边界与选型**:①**追踪不等于自动变好**——它只是**记录和比较**的基础设施,不会替你想出更好的模型;但它让你的实验**不白费**(每次尝试都沉淀成可查的知识),这是长期效率的复利。②**别追踪成负担**:记录该记的(关键参数、指标、模型、数据版本),别把每个中间变量都塞进去;团队要约定统一的实验命名和指标规范,否则追踪系统本身变成一团乱麻。③**选型**:**MLflow** 是开源事实标准、自托管友好、覆盖追踪+注册表+部署全流程;**Weights & Biases(W&B)** 在**可视化、团队协作、大模型训练的实时监控**上更强(深度学习/LLM 训练几乎是标配),但偏 SaaS;还有 Neptune、Comet 等。**DVC** 侧重数据和流水线版本,常和 MLflow 互补。④**和整个 MLOps 打通**才发挥最大价值:追踪的模型进注册表(版本化)→ CI/CD 质量门读它比较(22.7)→ 部署到生产(22.6)→ 监控漂移(22.10)→ 触发重训(回到追踪)。**结论:实验追踪用一个朴素的"记录一切"机制,把 ML 从不可复现的手工试错升级为可比较、可复现、可追溯的工程学科;它是 MLOps 的记忆中枢,和版本化一起构成可复现地基,支撑起质量门、回滚和持续训练的整个闭环。**

**English**:
1. **Experiment tracking's essence is astonishingly plain, yet without it the whole ML workflow collapses**: we reproduced MLflow's core in dozens of lines — merely "each run stores params, metrics, and the model in a structured place, queryable and comparable later." But this plain thing turns ML from an "alchemist's workshop" into "manageable engineering." Without it, you fall into the hell every data scientist has known: manually editing numbers in notebooks, results on sticky notes, no memory three weeks later of how the best model came to be, unable to reproduce it, teammates rerunning the same experiments… **Experiment tracking is key to turning "I tried a lot" into "I know what I tried, which was best, and why."**
2. **Its deeper value is "reproducibility," the baseline of science and engineering**: we verified that — because parameters were fully logged — three weeks later (even a different person) can **precisely rebuild that best model from the record**. Reproducibility sounds obvious but is where ML most easily fails: an unfixed random seed, silently changed data, an upgraded dependency, a forgotten preprocessing step… any one turns "last time's 0.92 into 0.89," and you can't tell improvement from noise. Experiment tracking + data versioning (DVC) + code version (git) + environment freezing (containers) form the four pillars of reproducibility — exactly the prerequisite for 22.7's quality gate (to "compare against a baseline," the baseline itself must be reproducible and trustworthy).
3. **Honest limits and tool choice**: ① **Tracking doesn't equal auto-improvement** — it's infrastructure for **recording and comparing**, not for inventing a better model; but it makes your experiments **not wasted** (every attempt accumulates as queryable knowledge), a compounding long-term efficiency. ② **Don't make tracking a burden**: log what matters (key params, metrics, model, data version), not every intermediate variable; teams need agreed experiment naming and metric conventions, else the tracking system itself becomes a mess. ③ **Tool choice**: **MLflow** is the open-source de-facto standard, self-hosting-friendly, covering tracking + registry + deployment; **Weights & Biases (W&B)** is stronger at **visualization, team collaboration, and real-time monitoring of large-model training** (nearly standard for deep learning/LLM training) but leans SaaS; also Neptune, Comet. **DVC** focuses on data and pipeline versioning, often complementing MLflow. ④ **Integrating with the whole MLOps flow** maximizes value: tracked models enter the registry (versioning) → the CI/CD quality gate reads them to compare (22.7) → deploy to production (22.6) → monitor drift (22.10) → trigger retraining (back to tracking). **Conclusion: experiment tracking uses a plain "log everything" mechanism to upgrade ML from irreproducible manual trial-and-error into a comparable, reproducible, traceable engineering discipline; it's MLOps's memory hub, forming the reproducibility foundation with versioning, and supporting the whole loop of quality gate, rollback, and continuous training.**

> 💼 **实战视角 / Practical angle**
> **中文**:实验追踪落地:①**每次训练都追踪**——`mlflow.start_run()` 记 params/metrics/model + 数据版本 + git commit;②**用 UI 比较** run(`mlflow ui`)找最优、看超参对指标的影响;③**模型注册表**:把选中的模型注册、打版本、管阶段(Staging→Production), 供 CI/CD 质量门(22.7)读取和回滚;④**可复现四件套**:固定种子 + DVC 钉数据版本 + git 钉代码 + 容器钉环境;⑤**打通 MLOps**:追踪→注册→CI/CD→部署→监控→重训闭环;⑥**大模型/深度学习**训练监控用 **W&B**(实时 loss 曲线、系统指标、超参扫描可视化)。**别做的**:手动记结果、不记数据版本、追踪不可复现的训练。面试金句:*"实验追踪(MLflow)记录每次 run 的参数/指标/模型/数据版本, 让实验可比较可复现可追溯, 解决'忘了那个好模型怎么来的'; 配模型注册表做版本化和阶段流转, 支撑质量门和回滚; 可复现要固定种子+钉数据/代码/环境; 深度学习训练监控常用 W&B; 它是 MLOps 闭环的记忆中枢。"*
> **English**: Experiment tracking in practice: ① **track every training** — `mlflow.start_run()` logs params/metrics/model + data version + git commit; ② **compare runs in the UI** (`mlflow ui`) to find the best and see hyperparameter effects on metrics; ③ **model registry**: register the chosen model, version it, manage stages (Staging→Production) for the CI/CD quality gate (22.7) to read and roll back; ④ **reproducibility quartet**: fix seeds + DVC to pin data version + git to pin code + container to pin environment; ⑤ **integrate MLOps**: track→register→CI/CD→deploy→monitor→retrain loop; ⑥ for **large-model/deep-learning** training monitoring use **W&B** (real-time loss curves, system metrics, hyperparameter-sweep visualization). **Don't**: manually record results, skip data versioning, or track non-reproducible training. Interview line: *"Experiment tracking (MLflow) logs each run's parameters/metrics/model/data version, making experiments comparable, reproducible, traceable, solving 'forgot how that good model came to be'; pair with a model registry for versioning and stage transitions, supporting the quality gate and rollback; reproducibility needs fixed seeds + pinned data/code/environment; deep-learning training monitoring often uses W&B; it's the memory hub of the MLOps loop."*

---
### 小结 / Summary
- **中文**:实验追踪=系统记录每次 run 的参数/指标/模型产物/数据版本, 让实验可比较、可复现、可追溯。
- **English**: Experiment tracking = systematically log each run's params/metrics/model artifacts/data version, making experiments comparable, reproducible, traceable.
- **中文**:MLflow=Tracking(记录+UI 比较)+Model Registry(版本化+阶段流转)+Projects+Models; 支撑 22.7 质量门与回滚。
- **English**: MLflow = Tracking (log + UI compare) + Model Registry (versioning + stage transitions) + Projects + Models; supports 22.7's quality gate and rollback.
- **中文**:可复现四件套:固定种子+DVC 数据版本+git 代码+容器环境; 深度学习训练监控常用 W&B。
- **English**: Reproducibility quartet: fixed seeds + DVC data version + git code + container environment; deep-learning training monitoring often uses W&B.
